# W2 실습 — 내 감자에 SLM 올려보기

**Potato LLM 2주차 (7/29)**

## 오늘 하는 것

| | 내용 |
|---|---|
| ① | 내 감자(또는 Colab)에 **SLM을 올려서 돌려본다** |
| ② | **시험지가 어떻게 생겼는지** 뜯어본다 |
| ③ | **내 업무 문항을 직접 만들어** 시험을 본다 |

**③이 오늘의 핵심입니다.** 리더보드 점수가 아니라 *내 업무 기준*으로 모델을 재는 것,
그게 이 과정의 세 번째 산출물인 **Potato Bench**의 출발점이에요.

## 쓰는 법

- 셀을 위에서부터 차례로 실행하세요. `Shift + Enter` 또는 셀 왼쪽 **▶** 버튼
- 코드를 짤 필요는 없습니다. 바꿀 곳에는 **`← 여기`** 표시가 되어 있어요
- 오류가 나면 **디스코드에 화면을 그대로 캡처해서** 올려주세요. **오류도 오늘의 실습 결과입니다**

## 준비물

**구글 계정 하나면 됩니다.** 오늘은 전원 **Colab**으로 진행해요.
설치할 것도, 내 컴퓨터 사양도 상관없습니다.


## 0. 준비

0-1부터 0-5까지 위에서부터 하나씩 실행하세요. 다 합쳐 3~5분입니다.

### 먼저 GPU를 켜주세요
상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 `T4 GPU` → 저장**

### 0-1. GPU 확인

In [ ]:
!nvidia-smi -L

### 0-2. Ollama 설치 `1~2분`

두 줄입니다. 첫 줄은 Colab에 없는 도구 두 개를 깝니다 — `zstd`(압축 해제), `lshw`(GPU 인식).

In [ ]:
!apt-get -qq install -y zstd lshw
!curl -fsSL https://ollama.com/install.sh | sh

### 0-3. 서버 켜기

`ollama serve` 를 백그라운드로 띄웁니다. **`Ollama is running`** 이 나오면 성공이에요.

In [ ]:
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5
!curl -s localhost:11434

### 0-4. 모델 받기 `2~3분 · 약 2GB`

In [ ]:
MODEL = "qwen2.5:3b"   # ← 다른 모델을 쓰려면 여기만 바꾸세요

!ollama pull {MODEL}

### 0-5. 말 걸어보기 — 여기까지 되면 준비 끝

모델에게 직접 물어봅니다. **한글 답이 나오면 성공입니다.**

이어서 나오는 표에서 두 칸만 보세요.
- **PROCESSOR** — `100% GPU` 면 잘 잡힌 겁니다. `100% CPU` 면 느리지만 실습은 됩니다
- **SIZE** — 이 모델이 차지한 메모리. 4장 결과에도 다시 나옵니다

In [ ]:
import json, urllib.request

req = urllib.request.Request(
    "http://localhost:11434/api/generate",
    json.dumps({"model": MODEL, "prompt": "안녕? 너를 한 문장으로 소개해줘",
                "stream": False}).encode(),
    {"Content-Type": "application/json"})

print(json.load(urllib.request.urlopen(req))["response"])

!ollama ps

## 1. 시험 볼 모델 선택

**0장에서 받은 모델이 자동으로 들어갑니다.** 그대로 두고 넘어가시면 돼요.

> 다른 모델로도 시험해보고 싶으면 아래 `MY_MODELS` 를 직접 바꾸세요.
> 여러 개를 넣으면 전부 시험 봅니다 — 예: `["qwen2.5:3b", "llama3.2:1b"]`


In [ ]:
# 0장에서 받은 모델이 기본값입니다. 다른 모델로 시험하려면 여기를 바꾸세요.
MY_MODELS = [globals().get("MODEL", "qwen2.5:3b")]   # ← 여기

print("시험 볼 모델:", MY_MODELS)

## 2. 시험지 (seed 벤치 v0)

모델에게 도구 4개(날씨·파일읽기·계산기·메일)를 주고 **문제 5개**를 냅니다. 도구마다 하나씩, 그리고 함정 하나.

각 문제는 **3요소**로 되어 있습니다. 개발자분들은 **테스트 케이스**를 떠올리시면 빠릅니다 — 입력이 있고, 기대하는 동작이 있고, 마지막에 assertion이 있는 구조예요:
1. **입력**: 지시문
2. **기대 동작**: 어떤 도구를 불러야 하나 (`tool`)
3. **검증 방법**: 인자가 맞는지 확인하는 코드 (`check`)

**Q5는 함정**입니다. 도구로 할 수 없는 요구인데 모델이 도구를 부르면 환각(hallucination)으로 오답 처리합니다.

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {"type": "object", "properties": {
            "city": {"type": "string", "description": "도시 이름 (영문)"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}},
            "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "파일 내용을 읽는다",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "파일 경로"}},
            "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "수식을 계산한다",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "계산할 수식, 예: 2+3*4"}},
            "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "send_email",
        "description": "이메일을 보낸다",
        "parameters": {"type": "object", "properties": {
            "to": {"type": "string"}, "subject": {"type": "string"},
            "body": {"type": "string"}},
            "required": ["to", "subject", "body"]}}},
]

CASES = [
    {"q": "서울의 현재 날씨를 섭씨로 알려줘.",
     "tool": "get_weather",
     "check": lambda a: "seoul" in str(a.get("city", "")).lower()},
    {"q": "report.csv 파일의 내용을 읽어줘.",
     "tool": "read_file",
     "check": lambda a: "report.csv" in str(a.get("path", ""))},
    {"q": "1847 곱하기 365는 얼마야? 도구를 써서 정확히 계산해.",
     "tool": "calculator",
     "check": lambda a: "1847" in str(a.get("expression", "")) and "365" in str(a.get("expression", ""))},
    {"q": "mentor@potato-llm.dev 에게 제목 '중간 공유', 내용 '8월 12일 오프라인에서 뵙겠습니다'로 메일 보내줘.",
     "tool": "send_email",
     "check": lambda a: a.get("to") == "mentor@potato-llm.dev" and len(str(a.get("subject", ""))) > 0 and len(str(a.get("body", ""))) > 0},
    # 함정: 주어진 도구로 할 수 없는 요구 → 도구를 부르면 환각
    {"q": "지금 KOSPI 지수 알려줘.",
     "tool": None, "check": lambda a: True},
]
print(f"시험지 준비 완료: {len(CASES)}문항 (함정 1개 포함)")


### 2-1. 모델에게 실제로 뭐가 들어가나

우리가 보내는 건 질문 한 줄이지만, 모델이 받는 건 그게 다가 아닙니다.
**도구 정의 4개가 통째로 같이 갑니다.** 그래야 모델이 "내가 뭘 부를 수 있는지" 알거든요.

아래를 실행하면 1번 문항이 실제로 어떤 모양으로 전달되는지 보입니다.

> 도구를 늘릴수록 이 입력이 커집니다. **5장에서 내 도구를 추가하면 여기도 같이 늘어나요.**

In [ ]:
payload = {
    "model": MY_MODELS[0],
    "messages": [{"role": "user", "content": CASES[0]["q"]}],
    "tools": TOOLS,
    "stream": False,
}

print(json.dumps(payload, ensure_ascii=False, indent=2))

## 3. 채점기

In [ ]:
import json, shutil, subprocess, time, urllib.request

OLLAMA = "http://localhost:11434"


def run_case(model, case):
    """문제 1개를 내고 채점한다. raw 에는 모델이 실제로 뱉은 응답이 그대로 들어간다."""
    body = json.dumps({
        "model": model, "stream": False,
        "messages": [{"role": "user", "content": case["q"]}],
        "tools": TOOLS,
    }).encode()
    t0 = time.time()
    req = urllib.request.Request(f"{OLLAMA}/api/chat", body, {"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        resp = json.load(r)
    dt = time.time() - t0
    msg = resp.get("message", {})
    calls = msg.get("tool_calls") or []
    tps = resp.get("eval_count", 0) / (resp.get("eval_duration", 1) / 1e9) if resp.get("eval_duration") else 0

    if case["tool"] is None:
        ok = len(calls) == 0
        detail = f"환각! 없는 기능에 도구 사용: {[c['function']['name'] for c in calls]}" if calls else "도구 안 씀 (정답)"
    elif not calls:
        ok, detail = False, "tool call 없음 (그냥 텍스트로 답변)"
    else:
        fn = calls[0]["function"]
        args = fn.get("arguments", {})
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except json.JSONDecodeError:
                return {"ok": False, "detail": f"인자 JSON 파싱 실패: {args[:80]}",
                        "tps": tps, "sec": dt, "raw": msg}
        ok = fn.get("name") == case["tool"] and case["check"](args)
        detail = f"{fn.get('name')}({json.dumps(args, ensure_ascii=False)[:100]})" + ("" if ok else " ← 오답")
    return {"ok": ok, "detail": detail, "tps": tps, "sec": dt, "raw": msg}


def peak_memory(model):
    """ollama ps 로 모델이 차지한 메모리를 읽는다 (모델이 로드된 직후 호출)."""
    if shutil.which("ollama") is None:
        return "?"          # 다른 스택을 쓰면 ollama ps 가 없다
    out = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
    for line in out.splitlines()[1:]:
        if line.split() and model.startswith(line.split()[0].split(":")[0]):
            parts = line.split()
            return " ".join(parts[2:4])
    return "?"
def show(i, case, r):
    """문항 하나의 결과를 출력한다 — 모델이 실제로 뭘 뱉었는지까지."""
    raw = r.get("raw") or {}
    said = (raw.get("content") or "").strip().replace("\n", " ")
    calls = json.dumps(raw.get("tool_calls") or [], ensure_ascii=False)
    print(f"[{'O' if r['ok'] else 'X'}] Q{i} {case['q'][:34]}")
    print(f"      · 말한 것 : {said[:90] or '(없음 — 도구만 불렀습니다)'}")
    print(f"      · 부른 것 : {calls[:120]}")
    print(f"      · 채  점 : {r['detail']}  ({r['sec']:.1f}s, {r['tps']:.0f} tok/s)")


print("채점기 준비 완료")

## 4. 시험 실행

In [ ]:
records = []
for model in MY_MODELS:
    print(f"\n{'='*70}\n모델: {model}\n{'='*70}")
    results = []
    for i, case in enumerate(CASES, 1):
        try:
            r = run_case(model, case)
        except Exception as e:
            r = {"ok": False, "detail": f"에러: {e}", "tps": 0, "sec": 0, "raw": {}}
        results.append(r)
        show(i, case, r)
    mem = peak_memory(model)
    n_ok = sum(r["ok"] for r in results)
    tps_list = [r["tps"] for r in results if r["tps"]]
    avg_tps = sum(tps_list) / max(1, len(tps_list))
    print(f"\n  ▶ 성공률 {n_ok}/{len(CASES)} · 평균 {avg_tps:.0f} tok/s · 메모리 {mem}")
    records.append({"model": model, "score": f"{n_ok}/{len(CASES)}", "tps": f"{avg_tps:.0f}", "mem": mem})

## 5. 내 업무 문항 만들기 ← 오늘의 핵심

2장에서 본 문항은 **날씨·파일·계산기·메일** 같은 일반적인 것이었습니다.
이제 **여러분 업무**로 바꿔볼 차례예요.

### 문항 하나에 들어가는 3요소

| | | 예시 |
|---|---|---|
| ① **입력** | 시키는 말 | "지난달 미수금 뽑아서 김부장님께 메일 보내줘" |
| ② **기대 동작** | 어떤 도구를 불러야 정답인가 | `send_email` |
| ③ **검증** | 자동으로 채점하는 방법 | `to` 에 부장님 주소가 들어갔는가 |

**③을 가장 많이 빠뜨립니다.** 사람이 눈으로 보고 판단하면 30명이 함께 쓸 수 없어요.
자동으로 채점돼야 100번이든 1000번이든 잴 수 있습니다.

> 이렇게 각자 만든 문항을 모으면 그게 **Potato Bench**입니다.
> 오늘은 **1~2개만** 만들어봐도 충분해요.

In [ ]:
# ① 내 업무에서 쓸 도구를 정의합니다
MY_TOOLS = [
    {"type": "function", "function": {
        "name": "search_docs",                          # ← 도구 이름
        "description": "사내 문서를 검색한다",             # ← 언제 쓰는 도구인지 (모델이 이 설명을 보고 판단합니다)
        "parameters": {"type": "object", "properties": {
            "keyword": {"type": "string", "description": "검색어"}},
            "required": ["keyword"]}}},
]

# ② 내 업무 문항을 씁니다
MY_CASES = [
    {"q": "작년 4분기 매출 보고서 찾아줘.",                          # ← ① 입력
     "tool": "search_docs",                                      # ← ② 기대 동작 (None 이면 함정 문항)
     "check": lambda a: "매출" in str(a.get("keyword", ""))},      # ← ③ 검증
]

# ③ 시험을 봅니다 (여러 번 실행해도 도구가 중복 추가되지 않습니다)
_have = {t["function"]["name"] for t in TOOLS}
TOOLS = TOOLS + [t for t in MY_TOOLS if t["function"]["name"] not in _have]

print(f"내 문항 {len(MY_CASES)}개로 {MY_MODELS[0]} 시험\n")
for i, case in enumerate(MY_CASES, 1):
    show(i, case, run_case(MY_MODELS[0], case))
    print()

## 6. 더 해보기

1. **모델을 바꿔서 다시 시험** — 1장의 `MY_MODELS` 를 바꾸고 4장을 다시 실행하세요.
   - `qwen2.5:3b` vs `qwen2.5-coder:3b` — 후자는 **도구 호출 학습이 빠져 있어** 점수가 급락합니다.
     "SLM 고를 때 **도구 호출 학습 여부**를 봐야 한다"는 게 이것입니다.
   - Colab이라면 먼저 `!ollama pull qwen2.5-coder:3b`
2. **실패한 문항은 왜 실패했나?** 4유형으로 분류해보세요 (W6에서 처방을 배웁니다):
   - **A.** JSON 문법이 깨짐 · **B.** 스키마 위반 · **C.** 환각(없는 도구·인자) · **D.** 도구를 아예 안 씀
3. **함정 문항을 만들어보세요** — 주어진 도구로 할 수 없는 걸 시켜서, 모델이 지어내는지 봅니다.

---

### 이건 해보시면 좋습니다

시간 나실 때 **내 업무 문항 한두 개**를 5장 형식으로 더 적어보세요.
안 하셔도 다음 주 진행에는 지장 없습니다. 다만 **직접 해보신 분이 W3에서 훨씬 재밌을 거예요** — 다음 주에 만들 에이전트가 볼 시험지가 그거거든요.

## 7. 참고 — 멘토 실측 결과 (2026-07-26)

**같은 5문항 시험지**로 멘토가 미리 돌려본 결과입니다. 성공률은 그대로 비교하시면 돼요.

> ⚠️ **속도(tok/s)는 비교하지 마세요.** 멘토는 맥북 로컬, 여러분은 Colab GPU라 하드웨어가 다릅니다.
> 같은 모델이라도 숫자가 크게 다르게 나옵니다.
상세 분석: [docs/08 실측 문서](../docs/08-20260727-0700-slm-stack-experiments.md)

| 모델 | 크기 | 성공률 | 속도(tok/s) | 비고 |
|---|---|---|---|---|
| qwen2.5:3b | 3B | 5/5 | 108 | **기본 권장** |
| gemma4 (E4B) | 4B | 5/5 | 67 | 만점이나 9.6GB |
| HyperCLOVA X SEED | 1.5B | 4/5* | 267 | 국산·속도 1위 · **이 노트북으론 안 됨** |
| qwen2.5:1.5b | 1.5B | 4/5 | 188 | **저사양 권장** |
| xLAM-1b-fc-r | 1B | 4/5* | 109 | 함수 호출 특화 · **이 노트북으론 안 됨** |
| Ministral-3 (Q4) | 3B | 4/5 | 102 | 양자화로 속도 2배 (fp16: 47) |
| llama3.2:1b | 1B | 2/5 | 173 | |
| phi4-mini / SmolLM3 / qwen2.5-coder | 3B급 | 1/5 | - | tool call 안 나옴 — 피할 것 |

\* 표시 두 모델은 **Ollama의 표준 도구 호출 API를 지원하지 않습니다.** 도구 정의를 프롬프트에 직접 심는 방식으로 따로 잰 값이라, 1장에서 이 모델을 골라 넣으면 **전부 실패로 나옵니다.**

관찰 3가지:
1. 같은 3B급인데 5/5 ~ 1/5 — **tool calling 학습 여부**가 벤치 점수보다 중요
2. 같은 GGUF 파일이 Ollama 5/5 vs llama.cpp 4/5 — **스택도 변수.** 결과를 남길 땐 어떤 실행기로 돌렸는지도 같이 적어야 한다
3. Ministral fp16→Q4: 점수 유지하고 속도 2배 — **양자화 예고편 (W4)**